# Model comparison results
This notebook loads evaluation results from multiple CSV files, processes them (including averaging over seeds if configured), and generates LaTeX tables for each task as well as a comprehensive recap table comparing all models across tasks. The configuration at the top allows you to specify which CSV files to load, whether to use seeded results, and which models, tasks, and metrics to include in the tables. The final LaTeX tables are saved in the "tables" directory.

## Expected CSV Input Format

All CSV files should contain the following columns:
- **model_name**: Name/identifier of the model being evaluated (e.g., "eve_v05", "mistralai/mistral-small-3.2-24b-instruct")
- **task**: Task name with n-shot specification (e.g., "open_ended_0_shot", "mcqa_single_answer_5_shot")
- **subtask**: Subtask identifier (can be used for different variations or domains)
- **metric**: Name of the metric being measured (e.g., "acc", "f1", "llm_as_judge_avg")
- **value**: Numeric value for the metric (typically in range [0, 1] for percentages)

### Example CSV rows:
```
model_name,task,subtask,metric,value
eve_v05,open_ended_0_shot,generation,llm_as_judge_avg,0.856
mistralai/mistral-small-3.2-24b-instruct,mcqa_single_answer_0_shot,mcqa,acc,0.723
```

### For Seeded Results:
When using `USE_SEEDED_RESULTS = True`, the task column should include seed information:
```
model_name,task,subtask,metric,value
eve_v05,open_ended_0_shot_seed_42,generation,llm_as_judge_avg,0.856
eve_v05,open_ended_0_shot_seed_123,generation,llm_as_judge_avg,0.863
```

The notebook will automatically extract seed numbers, average across seeds, and compute standard deviations.

In [ ]:
import pandas as pd
import re

# ============= CONFIGURATION =============
# Set to True to load and average seeded results
USE_SEEDED_RESULTS = False
# CSV files to load (will be concatenated)
CSV_FILES = ['new_oe_metrics.csv', 'scoring_big.csv', 'eve_api_v05.csv']
# Seeded results file (only used if USE_SEEDED_RESULTS = True)
SEEDED_CSV_FILE = 'seeded_gen.csv'
# =========================================

"""
CONFIGURATION GUIDE:

1. USE_SEEDED_RESULTS (bool):
   - False: Load regular CSV files and concatenate them
   - True: Load seeded results file, extract seeds from task names, 
     average metrics across seeds, and combine with regular CSV files
   
2. CSV_FILES (list of str):
   - List of CSV file paths to load and concatenate
   - All files must have the same column structure (model_name, task, subtask, metric, value)
   - Empty strings in the list will be ignored
   - Example: ['results_batch1.csv', 'results_batch2.csv']

3. SEEDED_CSV_FILE (str):
   - Path to CSV file containing seeded results
   - Only used when USE_SEEDED_RESULTS = True
   - Task names should follow pattern: "task_name_0_shot_seed_123"
   - The notebook will automatically:
     a) Extract seed numbers from task names
     b) Group by (model, task_no_seed, subtask, metric)
     c) Compute mean and std across seeds
     d) Filter out stderr metrics before averaging
"""

# Load data based on configuration
if USE_SEEDED_RESULTS:
    # Load seeded results
    df_seeded = pd.read_csv(SEEDED_CSV_FILE)
    print(f"Loaded seeded results: {len(df_seeded)} rows")
    
    # Function to extract seed from task name
    def extract_seed_info(task_name):
        # Pattern: "task_name_0_shot_seed_1234" -> ("task_name_0_shot", "1234")
        match = re.match(r'^(.+)_seed_(\d+)$', task_name)
        if match:
            return match.group(1), match.group(2)
        return task_name, None
    
    # Extract seed information
    df_seeded[['task_no_seed', 'seed']] = df_seeded['task'].apply(
        lambda x: pd.Series(extract_seed_info(x))
    )
    
    # Filter out stderr metrics before averaging
    metrics_to_exclude = df_seeded['metric'].str.contains('stderr', case=False, na=False)
    df_no_stderr = df_seeded[~metrics_to_exclude].copy()
    
    # Average over seeds: group by everything except seed
    # Compute both mean and std
    df_stats = df_no_stderr.groupby(
        ['model_name', 'task_no_seed', 'subtask', 'metric'],
        as_index=False
    ).agg(
        value_mean=('value', 'mean'),
        value_std=('value', 'std'),
        seed_count=('value', 'count')
    )
    
    # Create averaged dataframe with mean values
    df_averaged = df_stats[['model_name', 'task_no_seed', 'subtask', 'metric', 'value_mean']].copy()
    df_averaged = df_averaged.rename(columns={'value_mean': 'value', 'task_no_seed': 'task'})
    
    # Create std dataframe with std values
    df_std = df_stats[['model_name', 'task_no_seed', 'subtask', 'metric', 'value_std']].copy()
    df_std['metric'] = df_std['metric'] + '_std'
    df_std = df_std.rename(columns={'value_std': 'value', 'task_no_seed': 'task'})
    
    # Remove rows where std is NaN (single seed cases)
    df_std = df_std.dropna(subset=['value'])
    
    # Combine mean and std
    df_averaged_with_std = pd.concat([df_averaged, df_std], ignore_index=True)
    
    print(f"Averaged over seeds: {len(df_averaged)} mean rows + {len(df_std)} std rows")
    print(f"Seeds found: {sorted(df_seeded['seed'].dropna().unique())}")
    
    # Show seed counts per metric
    seed_counts = df_stats[['metric', 'seed_count']].drop_duplicates()
    print(f"\nSeed counts per metric:")
    for _, row in seed_counts.iterrows():
        print(f"  {row['metric']}: {int(row['seed_count'])} seeds")
    
    # Load non-seeded results and concatenate
    dfs_to_concat = [pd.read_csv(f) for f in CSV_FILES if f]
    df = pd.concat([df_averaged_with_std] + dfs_to_concat, ignore_index=True)
else:
    # Load regular results
    dfs_to_concat = [pd.read_csv(f) for f in CSV_FILES if f]
    df = pd.concat(dfs_to_concat, ignore_index=True)
    print(f"Loaded regular results: {len(df)} rows")

# Extract base task name and n-shot value from full task name
def extract_task_and_shot(task_name):
    # Pattern to match task names like "open_ended_5_shot"
    match = re.match(r'^(.+)_(\d+)_shot$', task_name)
    if match:
        return match.group(1), int(match.group(2))
    return task_name, None

# Add base_task and n_shot columns to the dataframe
df_extended = df.copy()
df_extended[['base_task', 'n_shot']] = df_extended['task'].apply(
    lambda x: pd.Series(extract_task_and_shot(x))
)

print(f"\nFinal dataset: {len(df_extended)} rows")
print(f"Models: {df_extended['model_name'].unique()}")

In [ ]:
df_extended

In [ ]:
# Drop all rows containing 'stderr' in the metric column
#df = df[~df['metric'].str.contains('stderr')]

# Remove all the rows that have as subtask mmlu_pro_*
#df = df[~df['subtask'].str.contains('mmlu_pro_')]

# Create a dictionary where the key is the task and the value is the corresponding subset of the df
task_dict = {task: df[df['task'] == task] for task in df['task'].unique()}

# Display the filtered dataframe
df.head()

In [ ]:
import os

"""
LATEX TABLE CONFIGURATION GUIDE:

This configuration controls which models, tasks, and metrics are included in the 
individual task tables and how they are displayed.

Parameters:
-----------
models (list or None):
    - List of model names to include in tables
    - Use None to include all models in the dataset
    - Example: ['eve_v05', 'mistralai/mistral-small-3.2-24b-instruct']

n_shots (list or None):
    - List of n-shot values to include (e.g., [0], [0, 5], [0, 2, 5])
    - Use None to include all n-shot values
    - Each value creates separate tables

tasks (list or None):
    - List of base task names to include
    - Use None to include all tasks from the 'metrics' dictionary
    - Example: ['open_ended', 'hallucination_detection']

metrics (dict):
    - Maps each task to the list of metrics to display
    - Key: base task name (e.g., 'hallucination_detection')
    - Value: list of metric names (e.g., ['f1', 'acc'])
    - Only these metrics will appear in the tables for each task

model_aliases (dict, optional):
    - Custom display names for models in tables
    - Key: original model name, Value: display name
    - If not provided, names are auto-formatted (capitalized, _ -> -, / -> space)

metric_aliases (dict, optional):
    - Custom display names for metrics in tables
    - Key: original metric name, Value: display name
    - If not provided, names are auto-formatted (capitalized, _ -> space)

Notes:
------
- For open_ended tasks, values are automatically normalized by 5
- Maximum values are automatically bolded in the tables
- Tables are saved as .tex files in the 'tables/' directory
"""

# Configuration for LaTeX table generation (matching plot_config)
latex_config = {
    'models': ['eve_v05', 'mistralai/mistral-small-3.2-24b-instruct', 'qwen/qwen3-30b-a3b-instruct-2507', 'meta-llama/llama-4-scout', 'google/gemma-3-27b-it'],
    'n_shots': [0],  # List of n-shot values to include
    'tasks': None,  # None means all tasks in metrics dict, or specify list like ['open_ended', 'hallucination_detection']
    'metrics': {
        'hallucination_detection': ['f1', 'acc'],
        'mcqa_multiple_answer': ['acc', 'IoU'],
        'mcqa_single_answer': ['acc'],
        'open_ended': ['llm_as_judge_avg'],
        'open_ended_w_context': ['llm_as_judge_avg'],
    },
    'model_aliases': {
        # Optional: specify custom display names for models
        # If not specified, model names will be auto-formatted (capitalized, _ replaced with -)
        'eve_v05': 'EVE-Instruct',
        'mistralai/mistral-small-3.2-24b-instruct': 'Mistral Small 3.2',
        'qwen/qwen3-30b-a3b-instruct-2507': 'Qwen3 30B',
        'meta-llama/llama-4-scout': 'Llama 4 Scout',
        'google/gemma-3-27b-it': 'Gemma 3 27B',
    },
    'metric_aliases': {
        # Optional: specify custom display names for metrics
        'f1': 'F1',
        'acc': 'Accuracy',
        'llm_as_judge_avg': 'LLM as Judge',
        'precision': 'Precision',
        'recall': 'Recall',
    }
}

# Create tables directory if it doesn't exist
os.makedirs('tables', exist_ok=True)

# Filter the dataframe based on configuration
filtered_df = df_extended.copy()

# Apply model filter
if latex_config['models'] is not None:
    filtered_df = filtered_df[filtered_df['model_name'].isin(latex_config['models'])]

# Apply n_shot filter
if latex_config['n_shots'] is not None:
    filtered_df = filtered_df[filtered_df['n_shot'].isin(latex_config['n_shots'])]

# Apply task filter (using base_task)
if latex_config['tasks'] is not None:
    filtered_df = filtered_df[filtered_df['base_task'].isin(latex_config['tasks'])]

# Create a dictionary where the key is the task and the value is the corresponding subset
filtered_task_dict = {task: filtered_df[filtered_df['task'] == task] for task in filtered_df['task'].unique()}

# For each task, create a LaTeX table
for task_name, task_df in filtered_task_dict.items():
    # Apply metric filter if specified for this task
    base_task = task_df['base_task'].iloc[0] if len(task_df) > 0 else None
    if base_task and base_task in latex_config['metrics']:
        task_df = task_df[task_df['metric'].isin(latex_config['metrics'][base_task])]
    
    if task_df.empty:
        continue
    
    # Remove 'task' and 'subtask' columns
    table_df = task_df.drop(columns=['task', 'subtask', 'base_task', 'n_shot']).copy()
    
    # Format model names: use aliases if provided, otherwise auto-format
    if 'model_aliases' in latex_config and latex_config['model_aliases']:
        table_df['model_name'] = table_df['model_name'].apply(
            lambda x: latex_config['model_aliases'].get(x, x.replace('_', '-').replace('/', ' ').title())
        )
    else:
        table_df['model_name'] = table_df['model_name'].apply(
            lambda x: x.replace('_', '-').replace('/', ' ').title()
        )
    
    # Format metric names: use aliases if provided, otherwise auto-format
    if 'metric_aliases' in latex_config and latex_config['metric_aliases']:
        table_df['metric'] = table_df['metric'].apply(
            lambda x: latex_config['metric_aliases'].get(x, x.replace('_', ' ').title())
        )
    else:
        table_df['metric'] = table_df['metric'].apply(
            lambda x: x.replace('_', ' ').title()
        )
    
    # Normalize by 5 for open_ended tasks (matching plot behavior)
    if base_task == 'open_ended' or base_task == 'open_ended_w_context':
        table_df['value'] = table_df['value'].astype(float) / 5
    
    # Convert value column to percentage with 2 decimal points
    table_df['value'] = (table_df['value'] * 100).round(2)
    
    # For each metric, find the maximum value and bold it
    for metric in table_df['metric'].unique():
        metric_mask = table_df['metric'] == metric
        max_value = table_df.loc[metric_mask, 'value'].max()
        
        # Apply bold formatting to the maximum value(s) for this metric
        table_df.loc[metric_mask, 'value'] = table_df.loc[metric_mask, 'value'].apply(
            lambda x: f'\\textbf{{{x:.2f}}}' if x == max_value else f'{x:.2f}'
        )
    
    # Rename columns to be capitalized
    table_df = table_df.rename(columns={
        'model_name': 'Model',
        'metric': 'Metric',
        'value': 'Value'
    })
    
    # Format caption: replace underscores with spaces and capitalize
    formatted_caption = task_name.replace('_', ' ').title()
    
    # Convert to LaTeX with caption
    latex_table = table_df.to_latex(
        index=False,
        caption=f'{formatted_caption}',
        label=f'tab:{task_name.replace("_", "-")}',
        position='htbp',
        escape=False
    )
    
    # Save to file
    filename = f'tables/{task_name}.tex'
    with open(filename, 'w') as f:
        f.write(latex_table)
    
    print(f'Saved {filename}')

print(f'\nTotal tables created: {len(filtered_task_dict)}')
print(f'Configuration applied:')
print(f'  Models: {latex_config["models"] if latex_config["models"] else "All"}')
print(f'  N-shots: {latex_config["n_shots"] if latex_config["n_shots"] else "All"}')
print(f'  Tasks: {latex_config["tasks"] if latex_config["tasks"] else "All"}')

In [ ]:
# Create a comprehensive recap table with configurable task metrics

import numpy as np

"""
RECAP TABLE CONFIGURATION GUIDE:

This configuration creates a comprehensive benchmark table that compares multiple models
across all evaluation tasks. It can also incorporate win rate data if available.

Parameters:
-----------
models (list):
    - List of model names to compare in the recap table
    - Order determines the order of rows in the table

n_shot (int):
    - Single n-shot value to use for the recap table
    - Example: 0 for zero-shot evaluation

eve_model (str):
    - Name of the EVE model to use as reference for win rate calculations
    - Only relevant if using win_rate metrics in task_metrics

model_order (list):
    - Explicit ordering of models in the table (same as 'models' parameter)
    - This allows different ordering than alphabetical

task_metrics (dict):
    - Maps each task to the list of metrics to include in recap table
    - Key: base task name (e.g., 'mcqa_multiple_answer')
    - Value: list of metric names (e.g., ['IoU', 'acc'])
    - Special metric 'alpaca_win_rate' requires win_rate CSV file

task_aliases (dict):
    - Custom display names for tasks in table headers
    - Key: task name, Value: display name
    - Example: 'mcqa_multiple_answer' -> 'MCQA Multiple'

metric_aliases (dict):
    - Custom display names for metrics in table headers
    - Key: metric name, Value: display name
    - Can include LaTeX formatting (e.g., 'WR$^\\dagger$')

model_aliases (dict):
    - Custom display names for models in table rows
    - Key: original model name, Value: display name

Notes:
------
- Win rate metrics require a 'win_rate_big.csv' file in the same directory
- The win_rate CSV should have columns: model_name, task, opponent, metric, value
- Maximum values in each column are automatically bolded
- The table includes a multi-level header (task names + metric names)
- Output is saved as 'tables/recap.tex'
"""

# recap_table_config = {
#     'models': ['google/gemma-3-27b-it', 'meta-llama/llama-4-scout',
#                'mistralai/mistral-small-3.2-24b-instruct',
#                'qwen/qwen3-30b-a3b-instruct-2507', 'eve_v05'],
#     'n_shot': 0,
#     'eve_model': 'eve_v05',  # The EVE model to use for win rate calculations
#     'model_order': ['google/gemma-3-27b-it', 'meta-llama/llama-4-scout',
#                     'mistralai/mistral-small-3.2-24b-instruct',
#                     'qwen/qwen3-30b-a3b-instruct-2507', 'eve_v05'],
#
#     # Task metrics configuration - specify which metrics to show for each task
#     'task_metrics': {
#         'mcqa_multiple_answer': ['IoU', 'acc'],
#         'mcqa_single_answer': ['acc'],
#         'hallucination_detection': ['f1'],
#         'open_ended': ['llm_as_judge_avg', 'alpaca_win_rate'],
#         'open_ended_w_context': ['llm_as_judge_avg', 'alpaca_win_rate'],
#     },
#
#     # Task display names
#     'task_aliases': {
#         'mcqa_multiple_answer': 'MCQA Multiple',
#         'mcqa_single_answer': 'MCQA Single',
#         'hallucination_detection': 'Hallucination',
#         'open_ended': 'Open-Ended',
#         'open_ended_w_context': 'Open-Ended w/ Context',
#     },
#
#     # Metric display names
#     'metric_aliases': {
#         'IoU': 'IoU',
#         'acc': 'Acc.',
#         'f1': 'F1',
#         'llm_as_judge_avg': 'LLM Judge',
#         'alpaca_win_rate': 'WR$^\\dagger$',
#     },
#
#     # Model display names
#     'model_aliases': {
#         'google/gemma-3-27b-it': 'Gemma 3 27B',
#         'meta-llama/llama-4-scout': 'Llama 4 Scout',
#         'mistralai/mistral-small-3.2-24b-instruct': 'Mistral Small 3.2',
#         'qwen/qwen3-30b-a3b-instruct-2507': 'Qwen3 30B',
#         'eve_v05': 'EVE-Instruct',
#     },
# }
recap_table_config = {
    'models': ['mistralai/mistral-medium-3.1', 'qwen/qwen3-235b-a22b-thinking-2507',
               'openai/gpt-4.1', 'eve_v05'],
    'n_shot': 0,
    'eve_model': 'eve_v05',  # The EVE model to use for win rate calculations
    'model_order': ['mistralai/mistral-medium-3.1', 'qwen/qwen3-235b-a22b-thinking-2507',
               'openai/gpt-4.1', 'eve_v05'],

    # Task metrics configuration - specify which metrics to show for each task
    'task_metrics': {
        'mcqa_multiple_answer': ['IoU', 'acc'],
        'mcqa_single_answer': ['acc'],
        'hallucination_detection': ['f1'],
        'open_ended': ['llm_as_judge_avg', 'alpaca_win_rate'],
        'open_ended_w_context': ['llm_as_judge_avg', 'alpaca_win_rate'],
    },

    # Task display names
    'task_aliases': {
        'mcqa_multiple_answer': 'MCQA Multiple',
        'mcqa_single_answer': 'MCQA Single',
        'hallucination_detection': 'Hallucination',
        'open_ended': 'Open-Ended',
        'open_ended_w_context': 'Open-Ended w/ Context',
    },

    # Metric display names
    'metric_aliases': {
        'IoU': 'IoU',
        'acc': 'Acc.',
        'f1': 'F1',
        'llm_as_judge_avg': 'LLM Judge',
        'alpaca_win_rate': 'WR$^\\dagger$',
    },

    # Model display names
    'model_aliases': {
        'openai/gpt-4.1': 'GPT 4.1',
        'mistralai/mistral-medium-3.1': 'Mistral Medium 3.1',
        'qwen/qwen3-235b-a22b-thinking-2507': 'Qwen3 235B',
        'eve_v05': 'EVE-Instruct',
    },
}

# Filter main evaluation data
filtered_recap_df = df_extended[
    (df_extended['model_name'].isin(recap_table_config['models'])) &
    (df_extended['n_shot'] == recap_table_config['n_shot'])
].copy()

# Load win rate data - use the alpaca version which has better structure
try:
    win_rate_df_full = pd.read_csv('win_rate_big.csv')
    has_win_rate_data = True
    print(f"Loaded win rate data: {len(win_rate_df_full)} rows")
    print(f"Columns: {win_rate_df_full.columns.tolist()}")
except FileNotFoundError:
    print("Warning: win_rate_scoring_alpaca.csv not found. Win rates will be set to ---")
    has_win_rate_data = False

# Create a nested dictionary: model -> task -> metric -> value
model_data_dict = {}

for model in recap_table_config['model_order']:
    model_data_dict[model] = {}
    
    model_data = filtered_recap_df[filtered_recap_df['model_name'] == model]
    
    for task, metrics in recap_table_config['task_metrics'].items():
        model_data_dict[model][task] = {}
        
        task_data = model_data[model_data['base_task'] == task]
        
        for metric in metrics:
            if metric == 'alpaca_win_rate':
                # Special handling for win rate
                if model == recap_table_config['eve_model']:
                    model_data_dict[model][task][metric] = None  # EVE vs itself
                elif has_win_rate_data:
                    # Get win rate from win_rate_scoring_alpaca.csv
                    # With the alpaca version, we can properly filter by task and opponent
                    eve_wr = win_rate_df_full[
                        (win_rate_df_full['model_name'] == recap_table_config['eve_model']) &
                        (win_rate_df_full['task'] == task) &  # Now this is the actual task name!
                        (win_rate_df_full['opponent'] == model) &  # And opponent is separate
                        (win_rate_df_full['metric'] == 'alpaca_win_rate')
                    ]
                    
                    if len(eve_wr) > 0:
                        # Get the win rate value
                        wr_value = eve_wr['value'].values[0]
                        model_data_dict[model][task][metric] = wr_value * 100
                    else:
                        print(f"Warning: No win rate found for {model} vs {recap_table_config['eve_model']} on task {task}")
                        model_data_dict[model][task][metric] = np.nan
                else:
                    model_data_dict[model][task][metric] = np.nan
            else:
                # Regular metric from evaluation data
                metric_data = task_data[task_data['metric'] == metric]
                
                if len(metric_data) > 0:
                    value = metric_data['value'].values[0]
                    
                    # Normalize by 5 for open_ended tasks (matching the plot behavior)
                    if (task == 'open_ended' or task == 'open_ended_w_context') and metric == 'llm_as_judge_avg':
                        value = value / 5
                    
                    # Convert to percentage
                    model_data_dict[model][task][metric] = value * 100
                else:
                    model_data_dict[model][task][metric] = np.nan

# Calculate max values for each task-metric combination (for bolding)
max_values = {}
for task, metrics in recap_table_config['task_metrics'].items():
    max_values[task] = {}
    for metric in metrics:
        values = []
        for model in recap_table_config['model_order']:
            val = model_data_dict[model][task][metric]
            if val is not None and not np.isnan(val):
                values.append(val)
        max_values[task][metric] = max(values) if values else np.nan

# Build the LaTeX table
latex_lines = []
latex_lines.append("\\begin{table}[htbp]")
latex_lines.append("\\caption{Model Performance Across Benchmark Tasks (0-Shot)}")
latex_lines.append("\\centering")
latex_lines.append("\\small")
latex_lines.append("\\label{tab:benchmark-results-0-shot}")

# Calculate total number of columns (1 for model names + sum of metrics for all tasks)
total_cols = 1 + sum(len(metrics) for metrics in recap_table_config['task_metrics'].values())
col_spec = "l" + "c" * (total_cols - 1)
latex_lines.append(f"\\begin{{tabular}}{{{col_spec}}}")
latex_lines.append("\\toprule")

# Build header row 1 (task names with multicolumn)
header1_parts = [""]
col_idx = 2  # Start from column 2 (column 1 is model names)
for task in recap_table_config['task_metrics'].keys():
    task_alias = recap_table_config['task_aliases'].get(task, task.replace('_', ' ').title())
    num_metrics = len(recap_table_config['task_metrics'][task])
    
    if num_metrics == 1:
        header1_parts.append(task_alias)
    else:
        header1_parts.append(f"\\multicolumn{{{num_metrics}}}{{c}}{{{task_alias}}}")

latex_lines.append(" & ".join(header1_parts) + " \\\\")

# Build cmidrule lines
cmidrule_parts = []
col_idx = 2
for task in recap_table_config['task_metrics'].keys():
    num_metrics = len(recap_table_config['task_metrics'][task])
    if num_metrics > 1:
        cmidrule_parts.append(f"\\cmidrule(lr){{{col_idx}-{col_idx + num_metrics - 1}}}")
    else:
        cmidrule_parts.append(f"\\cmidrule(lr){{{col_idx}-{col_idx}}}")
    col_idx += num_metrics

latex_lines.append(" ".join(cmidrule_parts))

# Build header row 2 (metric names)
header2_parts = ["Model"]
for task, metrics in recap_table_config['task_metrics'].items():
    for metric in metrics:
        metric_alias = recap_table_config['metric_aliases'].get(metric, metric.replace('_', ' ').title())
        header2_parts.append(metric_alias)

latex_lines.append(" & ".join(header2_parts) + " \\\\")
latex_lines.append("\\midrule")

# Build data rows
for model in recap_table_config['model_order']:
    model_alias = recap_table_config['model_aliases'].get(model, model)
    
    row_parts = [model_alias]
    
    for task, metrics in recap_table_config['task_metrics'].items():
        for metric in metrics:
            val = model_data_dict[model][task][metric]
            
            # Format the value
            if val is None:
                # Special case for EVE win rate
                formatted = "---"
            elif np.isnan(val):
                formatted = "---"
            else:
                # Check if this is the max value for this task-metric
                max_val = max_values[task][metric]
                is_max = not np.isnan(max_val) and abs(val - max_val) < 0.01
                
                formatted = f"{val:.2f}"
                if is_max:
                    formatted = f"\\textbf{{{formatted}}}"
            
            row_parts.append(formatted)
    
    latex_lines.append(" & ".join(row_parts) + " \\\\")

# Footer
latex_lines.append("\\bottomrule")

# Create footnote if win_rate is used
has_win_rate = any('win_rate' in metrics for metrics in recap_table_config['task_metrics'].values())
if has_win_rate:
    latex_lines.append(f"\\multicolumn{{{total_cols}}}{{l}}{{\\footnotesize $^\\dagger$WR: EVE-Instruct win rate (\\%) against each opponent.}}")

latex_lines.append("\\end{tabular}")
latex_lines.append("\\end{table}")

# Combine all lines
recap_latex = "\n".join(latex_lines)

# Save to file
os.makedirs('tables', exist_ok=True)
with open('tables/recap.tex', 'w') as f:
    f.write(recap_latex)

print('\n' + '='*60)
print('Saved tables/recap.tex with comprehensive benchmark table')
print('='*60)
print(f'\nConfiguration:')
print(f'  Tasks: {list(recap_table_config["task_metrics"].keys())}')
print(f'  Total columns: {total_cols}')
print(f'  Models: {len(recap_table_config["model_order"])}')
print('\nPreview of the LaTeX table:')
print(recap_latex)

In [ ]:
# Create a markdown report
md_content = []

# Add title and header
md_content.append("# Model Evaluation Results Report\n")
md_content.append(f"*Generated on: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}*\n\n")

# Add summary section
md_content.append("## Summary\n")
md_content.append(f"This report contains evaluation results for {len(df['model_name'].unique())} models across {len(task_dict)} different tasks.\n\n")
md_content.append("### Models Evaluated:\n")
for model in sorted(df['model_name'].unique()):
    md_content.append(f"- {model}\n")
md_content.append("\n")

# Add tasks section
md_content.append("### Tasks:\n")
for task in sorted(task_dict.keys()):
    md_content.append(f"- {task}\n")
md_content.append("\n---\n\n")

# Add detailed results for each task
for task_name in sorted(task_dict.keys()):
    task_df = task_dict[task_name]

    # Remove 'task' and 'subtask' columns
    table_df = task_df.drop(columns=['task', 'subtask']).copy()

    # Filter metrics for Hallucination Detection (F1) tasks
    if 'hallucination_detection' in task_name:
        table_df = table_df[table_df['metric'].isin(['f1', 'acc', 'precision', 'recall'])]

    # Convert value column to percentage with 2 decimal points
    table_df['value'] = (table_df['value'] * 100).round(2)

    # Find the best model (max value for each metric if multiple metrics)
    max_value = table_df['value'].max()
    best_idx = table_df['value'].idxmax()
    best_model = table_df.loc[best_idx, 'model_name']
    best_value = table_df.loc[best_idx, 'value']

    # Format the value column with bold for the maximum values
    table_df['value'] = table_df['value'].apply(
        lambda x: f'**{x:.2f}**' if x == max_value else f'{x:.2f}'
    )

    # Add task section
    md_content.append(f"## {task_name.replace('_', ' ').title()}\n\n")
    md_content.append(f"**Best Model:** {best_model} ({best_value:.2f}%)\n\n")

    # Convert to markdown table
    md_table = table_df.to_markdown(index=False)
    md_content.append(md_table)
    md_content.append("\n\n")

# Combine all content
full_md = ''.join(md_content)

# Save to markdown file
os.makedirs('tables', exist_ok=True)
with open('tables/evaluation_report.md', 'w') as f:
    f.write(full_md)

print('Saved tables/evaluation_report.md')
print(f'\nReport includes {len(task_dict)} tasks and {len(df["model_name"].unique())} models')

# Plot Configuration and Visualizations

This section creates bar plots comparing models across tasks. The plot configurations allow you to:
- Select which models to compare
- Choose the n-shot setting (0, 2, 5, etc.)
- Specify which metric to use for each task
- Customize display names for models

## Plot Types Generated:
1. **Standard Bar Plot**: Side-by-side comparison of all models across tasks
2. **Baseline Comparison Plot**: Shows performance relative to a baseline model with floating bars

In [ ]:
df_extended['model_name'].unique()

In [ ]:
# Configurable bar plot for model comparison across tasks
import matplotlib.pyplot as plt
import numpy as np


color_palettes = ['#003145', '#006762', '#008e7a', '#00ae9d', '#335e6f', '#980000']

# # Configuration dictionary
# baseline_plot_config = {
#     'models': ['mistralai/mistral-small-3.2-24b-instruct', 'eve-esa/eve_v0.1', './shipmodel1230'],
#     'n_shot': 0,  # Choose 0, 2, or 5
#     'task_metrics': {
#         # Specify which metric to use for each task
#         'open_ended': 'llm_as_judge',
#         'open_ended_w_context': 'llm_as_judge',
#         'hallucination_detection': 'f1',
#         'mcqa_multiple_answer': 'acc',
#         'mcqa_single_answer': 'acc',
#     },
#     'tasks': None,  # None means all tasks in task_metrics, or specify list like ['open_ended', 'hallucination_detection']
# }
# Configuration dictionary
# plot_config = {
#     'models': ['eve_v05', 'mistralai/mistral-small-3.2-24b-instruct', 'qwen/qwen3-30b-a3b-instruct-2507', 'meta-llama/llama-4-scout', 'google/gemma-3-27b-it'],
#     'n_shot': 0,  # Choose 0, 2, or 5
#     'task_metrics': {
#         # Specify which metric to use for each task
#         'hallucination_detection': 'f1',
#         'mcqa_multiple_answer': 'acc',
#         'mcqa_single_answer': 'acc',
#         'open_ended': 'llm_as_judge_avg',
#         'open_ended_w_context': 'llm_as_judge_avg',
#     },
#     'tasks': None,  # None means all tasks in task_metrics, or specify list like ['open_ended', 'hallucination_detection']
# }
#
# # Model name aliases for display
# model_aliases = {
#     'mistralai/mistral-small-3.2-24b-instruct': 'Mistral Small 3.2',
#     'eve-api': 'EVE System',
#     'eve_v05': 'EVE 0.5'
# }


plot_config = {
    'models': ['eve_v05', 'mistralai/mistral-medium-3.1', 'openai/gpt-4.1', 'deepseek/deepseek-r1-0528', 'qwen/qwen3-235b-a22b-thinking-2507', 'eve-api-05'],
    'n_shot': 0,  # Choose 0, 2, or 5
    'task_metrics': {
        # Specify which metric to use for each task
        'hallucination_detection': 'f1',
        'mcqa_multiple_answer': 'acc',
        'mcqa_single_answer': 'acc',
        'open_ended': 'llm_as_judge_avg',
        'open_ended_w_context': 'llm_as_judge_avg',
    },
    'tasks': None,  # None means all tasks in task_metrics, or specify list like ['open_ended', 'hallucination_detection']
}

# Model name aliases for display
model_aliases = {
    'mistralai/mistral-small-3.2-24b-instruct': 'Mistral Small 3.2',
    'eve-api': 'EVE System',
    'eve_v05': 'EVE 0.5'
}

# Filter tasks based on config
tasks_to_plot = plot_config['tasks'] if plot_config['tasks'] is not None else list(plot_config['task_metrics'].keys())

# Collect data for each task with its specific metric
plot_data_list = []

for task in tasks_to_plot:
    if task not in plot_config['task_metrics']:
        print(f"Warning: No metric specified for task '{task}', skipping...")
        continue
    
    metric = plot_config['task_metrics'][task]
    # Filter data for this specific task and metric
    task_data = df_extended[
        (df_extended['model_name'].isin(plot_config['models'])) &
        (df_extended['base_task'] == task) &
        (df_extended['n_shot'] == plot_config['n_shot']) &
        (df_extended['metric'] == metric)
    ].copy()
    
    if not task_data.empty:
        task_data['task_metric'] = f"{task} ({metric})"
        plot_data_list.append(task_data)

    if task == 'open_ended' or task == 'open_ended_w_context':
    # Normalize by 5
        task_data['value'] = task_data['value'].astype(float) / 5

if not plot_data_list:
    print("No data to plot with the current configuration!")
else:
    # Combine all task data
    plot_data = pd.concat(plot_data_list, ignore_index=True)
    
    # Pivot data: rows = tasks, columns = models
    pivot_plot = plot_data.pivot_table(
        index='task_metric',
        columns='model_name',
        values='value',
        aggfunc='first'
    )
    
    # Reorder columns to match the order in config
    pivot_plot = pivot_plot[[col for col in plot_config['models'] if col in pivot_plot.columns]]
    
    # Convert to percentage
    pivot_plot = pivot_plot * 100
    
    # Create the bar plot with dynamic width based on number of tasks
    n_tasks = len(pivot_plot.index)
    n_models = len(pivot_plot.columns)
    fig_width = max(16, n_tasks * 1.5)  # Dynamic width based on number of tasks
    
    fig, ax = plt.subplots(figsize=(fig_width, 8))
    
    # Set up bar positions
    tasks = pivot_plot.index
    x = np.arange(len(tasks))
    width = 0.15  # Thinner bars to fit more models per group
    
    # Plot bars for each model using the custom color palette
    colors = [color_palettes[i % len(color_palettes)] for i in range(n_models)]
    
    for i, model in enumerate(pivot_plot.columns):
        offset = (i - n_models/2 + 0.5) * width
        bars = ax.bar(x + offset, pivot_plot[model], width, 
                       label=model_aliases.get(model, model),
                       color=colors[i],
                       edgecolor='black',
                       linewidth=0.5)
        
        # Add value labels on top of bars
        for bar in bars:
            height = bar.get_height()
            if not np.isnan(height):
                ax.text(bar.get_x() + bar.get_width()/2., height,
                       f'{height:.2f}',
                       ha='center', va='bottom', fontsize=7, rotation=0)
    
    # Customize plot
    ax.set_xlabel('Task (Metric)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Score (%)', fontsize=12, fontweight='bold')
    ax.set_title(f'Model Comparison Across Tasks ({plot_config["n_shot"]}-shot)', 
                 fontsize=14, fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels([task.replace('_', ' ').title() for task in tasks], rotation=45, ha='right', fontsize=9)
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    # Set y-axis to start from 0
    ax.set_ylim(bottom=0)
    
    plt.tight_layout()
    plt.savefig('tables/model_comparison_barplot.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f'Saved tables/model_comparison_barplot.png')
    print(f'Showing {len(tasks)} tasks and {n_models} models')
    print(f'Figure width: {fig_width} inches')

In [ ]:
# Baseline Comparison Plot - Floating bars from baseline
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle

# Configuration for baseline comparison plot
baseline_plot_config = {
    'baseline_model': 'mistralai/mistral-small-3.2-24b-instruct',  # This model's scores will be the baseline (starting point)
    'comparison_models': ['eve_v05',
                          'qwen/qwen3-30b-a3b-instruct-2507',
                          'meta-llama/llama-4-scout',
                          'google/gemma-3-27b-it'],
    'n_shot': 0,  # Choose 0, 2, or 5
    'task_metrics': {
        # Specify which metric to use for each task
        'hallucination_detection': 'f1',
        'mcqa_multiple_answer': 'acc',
        'mcqa_single_answer': 'acc',
        'open_ended': 'llm_as_judge_avg',
        'open_ended_w_context': 'llm_as_judge_avg',
    },
    'tasks': None,  # None means all tasks in task_metrics
    'model_aliases': {
        'eve_v05': 'EVE-Instruct',
        'mistralai/mistral-small-3.2-24b-instruct': 'Mistral Small 3.2',
        'qwen/qwen3-30b-a3b-instruct-2507': 'Qwen3 30B',
        'meta-llama/llama-4-scout': 'Llama 4 Scout',
        'google/gemma-3-27b-it': 'Gemma 3 27B',
    },
    'task_aliases': {
        'hallucination_detection': 'Halluc. Det. (F1)',
        'mcqa_multiple_answer': 'MCQA Multi (Acc)',
        'mcqa_single_answer': 'MCQA Single (Acc)',
        'open_ended': 'Open-Ended (LLM Judge)',
        'open_ended_w_context': 'Open-Ended w/ Ctx (LLM Judge)',
    }
}

color_palette = ['#006762', '#008e7a', '#00ae9d', '#335e6f']

# Filter tasks based on config
tasks_to_plot = baseline_plot_config['tasks'] if baseline_plot_config['tasks'] is not None else list(baseline_plot_config['task_metrics'].keys())

# Collect data for each task
all_models = [baseline_plot_config['baseline_model']] + baseline_plot_config['comparison_models']
task_data_dict = {}

for task in tasks_to_plot:
    if task not in baseline_plot_config['task_metrics']:
        print(f"Warning: No metric specified for task '{task}', skipping...")
        continue

    metric = baseline_plot_config['task_metrics'][task]

    # Get data for all models for this task
    task_data = df_extended[
        (df_extended['model_name'].isin(all_models)) &
        (df_extended['base_task'] == task) &
        (df_extended['n_shot'] == baseline_plot_config['n_shot']) &
        (df_extended['metric'] == metric)
    ].copy()

    if not task_data.empty:
        # Normalize by 5 for open_ended tasks
        if task == 'open_ended' or task == 'open_ended_w_context':
            task_data['value'] = task_data['value'].astype(float) / 5

        # Store as dictionary: model -> value
        task_values = {}
        for _, row in task_data.iterrows():
            task_values[row['model_name']] = row['value']

        task_data_dict[task] = task_values

if not task_data_dict:
    print("No data to plot with the current configuration!")
else:
    # Create the plot
    n_tasks = len(task_data_dict)
    n_comparison_models = len(baseline_plot_config['comparison_models'])

    fig, ax = plt.subplots(figsize=(max(14, n_tasks * 2.5), 8))

    # Set up positions
    x = np.arange(n_tasks)
    width = 0.15  # Width of each bar

    # For each task, plot the baseline and comparison bars
    for task_idx, (task, model_values) in enumerate(task_data_dict.items()):
        # Get baseline value
        baseline_value = model_values.get(baseline_plot_config['baseline_model'], np.nan)

        if np.isnan(baseline_value):
            print(f"Warning: No baseline value for task '{task}'")
            continue

        # Plot comparison models
        for comp_idx, comp_model in enumerate(baseline_plot_config['comparison_models']):
            comp_value = model_values.get(comp_model, np.nan)

            if np.isnan(comp_value):
                continue

            # Calculate bar position and height
            offset = (comp_idx - n_comparison_models/2 + 0.5) * width
            x_pos = task_idx + offset

            # Determine if this is better (+) or worse (-) than baseline
            # Use consistent color for each model regardless of performance
            color = color_palette[comp_idx % len(color_palette)]
            
            if comp_value >= baseline_value:
                # Bar goes UP from baseline
                bar_bottom = baseline_value
                bar_height = comp_value - baseline_value
                alpha = 0.8
            else:
                # Bar goes DOWN from baseline
                bar_bottom = comp_value
                bar_height = baseline_value - comp_value
                alpha = 0.6  # Slightly more transparent for worse performance

            # Draw the bar
            bar = ax.bar(x_pos, bar_height, width, bottom=bar_bottom,
                        color=color, alpha=alpha, edgecolor='black', linewidth=0.8,
                        label=baseline_plot_config['model_aliases'].get(comp_model, comp_model) if task_idx == 0 else "")

            # Add value label at the top of the bar
            label_y = comp_value
            label_text = f'{comp_value*100:.2f}'
            ax.text(x_pos, label_y, label_text,
                   ha='center', va='bottom' if comp_value >= baseline_value else 'top',
                   fontsize=7, rotation=0)

    # Draw horizontal lines at baseline values for each task
    for task_idx, (task, model_values) in enumerate(task_data_dict.items()):
        baseline_value = model_values.get(baseline_plot_config['baseline_model'], np.nan)
        if not np.isnan(baseline_value):
            # Draw a thick horizontal line at baseline
            ax.hlines(baseline_value, task_idx - 0.4, task_idx + 0.4,
                     colors='#003145', linewidth=3, linestyles='solid',
                     label=f'Baseline ({baseline_plot_config["model_aliases"].get(baseline_plot_config["baseline_model"], baseline_plot_config["baseline_model"])})' if task_idx == 0 else "")

            # Add baseline value label
            ax.text(task_idx - 0.45, baseline_value, f'{baseline_value*100:.2f}',
                   ha='right', va='center', fontsize=8, fontweight='bold',
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#003145', linewidth=1.5))

    # Customize plot
    ax.set_xlabel('Task', fontsize=13, fontweight='bold')
    ax.set_ylabel('Score (0-1)', fontsize=13, fontweight='bold')
    ax.set_title(f'Model Performance Relative to Baseline ({baseline_plot_config["n_shot"]}-shot)\n' +
                 f'Baseline: {baseline_plot_config["model_aliases"].get(baseline_plot_config["baseline_model"], baseline_plot_config["baseline_model"])}',
                 fontsize=14, fontweight='bold', pad=20)

    # Set x-axis labels
    ax.set_xticks(x)
    task_labels = [baseline_plot_config['task_aliases'].get(task, task.replace('_', ' ').title())
                   for task in task_data_dict.keys()]
    ax.set_xticklabels(task_labels, rotation=45, ha='right', fontsize=10)

    # Set y-axis limits
    ax.set_ylim(0, 1)
    ax.set_yticks(np.arange(0, 1.1, 0.1))
    ax.set_yticklabels([f'{int(y*100)}%' for y in np.arange(0, 1.1, 0.1)], fontsize=9)

    # Add grid
    ax.grid(axis='y', alpha=0.3, linestyle='--', zorder=0)

    # Legend
    handles, labels = ax.get_legend_handles_labels()
    # Remove duplicate labels
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(),
             loc='upper left', bbox_to_anchor=(1, 1), fontsize=10,
             title='Models', title_fontsize=11)

    plt.tight_layout()
    plt.savefig('tables/baseline_comparison_plot.png', dpi=300, bbox_inches='tight')
    plt.show()

    print(f'Saved tables/baseline_comparison_plot.png')
    print(f'Showing {len(task_data_dict)} tasks')
    print(f'Baseline model: {baseline_plot_config["baseline_model"]}')
    print(f'Comparison models: {len(baseline_plot_config["comparison_models"])}')


## Win rates table

This section analyzes win rate data from pairwise LLM comparisons. 

### Expected Win Rate CSV Format:
The win_rate CSV file should contain:
- **model_name**: The model being evaluated
- **task**: Task name or opponent model identifier
- **opponent**: The opponent model (if using separate opponent column)
- **subtask**: Subtask identifier (e.g., "win_rate")
- **metric**: Metric name (e.g., "alpaca_win_rate", "total_wins", "total_ties")
- **value**: Numeric value for the metric

### Win Rate Metrics:
- **alpaca_win_rate**: Win rate calculated as (wins + 0.5 * ties) / total
- **total_wins**: Number of times the model won
- **total_ties**: Number of ties
- **total_evaluations**: Total number of comparisons
- **win_rate_judge_X**: Win rate according to specific judge X

In [ ]:
# Win Rate Analysis Section

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Load win rate data
win_rate_df = pd.read_csv('win_rate_scoring.csv')

# Configuration for win rate analysis
win_rate_config = {
    'models': ['eve_v04', 'eve-api'],  # Models to analyze (None for all)
    'opponents': None,  # Opponent models to compare against (None for all)
    'metrics': {
        # Specify which metrics to include in tables
        'primary': ['avg_win_rate', 'alpaca_win_rate', 'avg_accuracy', 'total_wins', 'total_ties'],
        'judges': ['win_rate_judge_mistral-large-2512', 'win_rate_judge_gpt-4.1-mini', 
                   'win_rate_judge_qwen3-235b-a22b-2507', 'win_rate_judge_deepseek-v3.2'],
        'additional': ['unanimous_decisions_rate', 'majority_wins_rate', 'win_rate_difference_vs_opponent']
    },
    'model_aliases': {
        'eve_v04': 'EVE v0.4',
        'eve-api': 'EVE API',
        'mistralai/mistral-small-3.2-24b-instruct': 'Mistral Small 3.2',
        'qwen/qwen3-30b-a3b-instruct-2507': 'Qwen3 30B',
        'meta-llama/llama-4-scout': 'Llama 4 Scout',
        'google/gemma-3-27b-it': 'Gemma 3 27B',
    },
    'metric_aliases': {
        'avg_win_rate': 'Avg Win Rate',
        'alpaca_win_rate': 'Alpaca Win Rate',
        'avg_accuracy': 'Avg Accuracy',
        'total_wins': 'Total Wins',
        'total_ties': 'Total Ties',
        'total_evaluations': 'Total Evaluations',
        'win_rate_difference_vs_opponent': 'Win Rate Diff',
        'accuracy_difference_vs_opponent': 'Accuracy Diff',
        'unanimous_decisions_rate': 'Unanimous Decisions',
        'majority_wins_rate': 'Majority Wins',
        'avg_position_bias_difference': 'Position Bias',
    }
}

# Filter data based on configuration
filtered_win_rate_df = win_rate_df.copy()

if win_rate_config['models'] is not None:
    filtered_win_rate_df = filtered_win_rate_df[filtered_win_rate_df['model_name'].isin(win_rate_config['models'])]

if win_rate_config['opponents'] is not None:
    filtered_win_rate_df = filtered_win_rate_df[filtered_win_rate_df['task'].isin(win_rate_config['opponents'])]

print(f"Win rate data loaded: {len(filtered_win_rate_df)} rows")
print(f"Models: {filtered_win_rate_df['model_name'].unique()}")
print(f"Opponents: {filtered_win_rate_df['task'].unique()}")
print(f"Metrics: {filtered_win_rate_df['metric'].unique()}")

In [ ]:
# Compute AlpacaEval-style win rate
# AlpacaEval win rate = (model_wins + 0.5 * ties) / total

# Get the necessary metrics for each model-opponent pair
alpaca_win_rate_rows = []

for model in filtered_win_rate_df['model_name'].unique():
    for opponent in filtered_win_rate_df['task'].unique():
        # Get data for this model-opponent pair
        pair_data = filtered_win_rate_df[
            (filtered_win_rate_df['model_name'] == model) & 
            (filtered_win_rate_df['task'] == opponent)
        ]
        
        # Extract required metrics
        total_wins_row = pair_data[pair_data['metric'] == 'total_wins']
        total_ties_row = pair_data[pair_data['metric'] == 'total_ties']
        total_evals_row = pair_data[pair_data['metric'] == 'total_evaluations']
        
        if not total_wins_row.empty and not total_ties_row.empty and not total_evals_row.empty:
            total_wins = total_wins_row['value'].values[0]
            total_ties = total_ties_row['value'].values[0]
            total_evaluations = total_evals_row['value'].values[0]
            
            # Calculate alpaca_win_rate for the model (not opponent)
            alpaca_win_rate = (total_wins + 0.5 * total_ties) / total_evaluations
            
            # Create new row
            new_row = {
                'model_name': model,
                'task': opponent,
                'subtask': 'win_rate',
                'metric': 'alpaca_win_rate',
                'value': alpaca_win_rate
            }
            alpaca_win_rate_rows.append(new_row)

# Convert to DataFrame and append to filtered_win_rate_df
if alpaca_win_rate_rows:
    alpaca_df = pd.DataFrame(alpaca_win_rate_rows)
    filtered_win_rate_df = pd.concat([filtered_win_rate_df, alpaca_df], ignore_index=True)
    
    print(f"Added {len(alpaca_win_rate_rows)} alpaca_win_rate metrics")
    print(f"Updated filtered_win_rate_df now has {len(filtered_win_rate_df)} rows")
    
    # Show sample of alpaca_win_rate values
    print("\nSample alpaca_win_rate values:")
    print(filtered_win_rate_df[filtered_win_rate_df['metric'] == 'alpaca_win_rate'][['model_name', 'task', 'value']].head(10))

# Win Rate Evaluation

This section analyzes win rate data from pairwise model comparisons. 

## Configuration Options

The `win_rate_config` dictionary allows you to customize:

- **models**: List of models to analyze (None for all)
- **opponents**: List of opponent models to compare against (None for all)
- **metrics**: Grouped metrics for different purposes
  - `primary`: Main metrics for tables (win rate, accuracy, wins, ties)
  - `judges`: Per-judge win rates
  - `additional`: Additional analysis metrics
- **model_aliases**: Custom display names for models
- **metric_aliases**: Custom display names for metrics

## Visualizations Generated

1. **Heatmap**: Pairwise win rate matrix showing which models perform better against which opponents
2. **Radar Chart**: Multi-dimensional performance comparison across key metrics

## LaTeX Tables Generated

1. **Individual Tables**: One table per model-opponent pair with detailed metrics
2. **Summary Table**: Aggregated performance across all opponents

In [ ]:
# Heatmap: Win Rate Matrix
# Create a pairwise win rate heatmap

# Filter for avg_win_rate metric
win_rate_matrix_df = filtered_win_rate_df[filtered_win_rate_df['metric'] == 'alpaca_win_rate'].copy()

# Apply model aliases
if 'model_aliases' in win_rate_config and win_rate_config['model_aliases']:
    win_rate_matrix_df['model_name'] = win_rate_matrix_df['model_name'].apply(
        lambda x: win_rate_config['model_aliases'].get(x, x)
    )
    win_rate_matrix_df['task'] = win_rate_matrix_df['task'].apply(
        lambda x: win_rate_config['model_aliases'].get(x, x)
    )

# Convert to matrix format
pivot_matrix = win_rate_matrix_df.pivot_table(
    index='model_name',
    columns='task',
    values='value',
    aggfunc='first'
)

# Convert to percentage
pivot_matrix = pivot_matrix * 100

# Create the heatmap
fig, ax = plt.subplots(figsize=(12, 8))

# Use a diverging colormap centered at 50%
sns.heatmap(
    pivot_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',  # Red for low win rates, green for high
    center=50,
    vmin=0,
    vmax=100,
    cbar_kws={'label': 'Win Rate (%)'},
    linewidths=0.5,
    linecolor='gray',
    ax=ax
)

ax.set_title('Pairwise Win Rate Heatmap\n(Row model vs Column opponent)', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Opponent Model', fontsize=12, fontweight='bold')
ax.set_ylabel('Evaluated Model', fontsize=12, fontweight='bold')

# Rotate labels for better readability
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.setp(ax.get_yticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('tables/win_rate_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print('Saved tables/win_rate_heatmap.png')

In [ ]:
# Summary Tables: One table per main model showing win rate against each opponent

# Filter for avg_win_rate metric only
summary_data = filtered_win_rate_df[filtered_win_rate_df['metric'] == 'win_rate_difference_vs_opponent'].copy()

# Apply aliases
if 'model_aliases' in win_rate_config and win_rate_config['model_aliases']:
    summary_data['model_name'] = summary_data['model_name'].apply(
        lambda x: win_rate_config['model_aliases'].get(x, x)
    )
    summary_data['task'] = summary_data['task'].apply(
        lambda x: win_rate_config['model_aliases'].get(x, x.replace('_', '-').replace('/', ' ').title())
    )

# Get unique main models
main_models = summary_data['model_name'].unique()

# Create one table per main model
for main_model in main_models:
    # Filter data for this main model
    model_data = summary_data[summary_data['model_name'] == main_model].copy()
    
    # Select only opponent (task) and win rate columns
    table_df = model_data[['task', 'value']].copy()
    
    # Convert to percentages
    table_df['value'] = (table_df['value'] * 100).round(2)
    
    # Sort by win rate descending
    table_df = table_df.sort_values('value', ascending=False)
    
    # Find max value and bold it
    max_val = table_df['value'].max()
    table_df['value'] = table_df['value'].apply(
        lambda x: f'\\textbf{{{x:.2f}}}' if x == max_val else f'{x:.2f}'
    )
    
    # Rename columns
    table_df = table_df.rename(columns={
        'task': 'Model',
        'value': 'Win Rate'
    })
    
    # Create caption and filename
    caption = f'{main_model} Win Rate vs Opponents'
    filename_safe = main_model.replace(' ', '_').replace('.', '')
    
    # Convert to LaTeX
    latex_table = table_df.to_latex(
        index=False,
        caption=caption,
        label=f'tab:winrate-summary-{filename_safe}',
        position='htbp',
        escape=False
    )
    
    # Save to file
    filename = f'tables/win_rate_summary_{filename_safe}.tex'
    with open(filename, 'w') as f:
        f.write(latex_table)
    
    print(f'Saved {filename}')
    print(f'\n{main_model} Summary Table:')
    print(table_df.to_string(index=False))
    print('\n' + '='*50 + '\n')

print(f'Total summary tables created: {len(main_models)}')